# Bell-state tomography: figures and analysis

This notebook collects the figures and the reasoning behind the repeated Bell-state tomography
of Q1–Q2 (SNL315). Every number is recomputed from the raw quantify datasets (`CD1/`) and the
CSV files in this folder; the heavy lifting lives in `bell_analysis.py` next to this notebook,
so the cells below only load, compute and plot.

**What the experiment is.** Each *run* prepares $|\Psi^+\rangle = (|01\rangle + |10\rangle)/\sqrt2$ with an
$X_\pi$ on Q1 followed by a $\sqrt{i\mathrm{SWAP}}$, then measures it in 9 tomography settings
(4000 shots each in the newest set). Right before every run the notebook loop also measures a
fresh readout confusion matrix (CM). A *set* is ~50 such runs taken back to back and saved
as `Bell_state_<date>[_n].csv` (the per-run MLE density matrices).

**What the notebook shows.**

1. How $\rho$ is reconstructed, and the pooled density matrix of the best set.
2. Why runs must be **pooled before the MLE**, not averaged afterwards.
3. A simulation proving which fidelity estimator is unbiased.
4. The fidelity of every set, with both estimators.
5. Gate-calibration errors and the full error budget from a physical model.
6. What the confusion matrix really contains, and a check of the CM-recovery method.
7. Run-to-run stability against the shot-noise expectation.
8. Coherence times measured around the Bell sets.

**Running it.** Use the **qblox_env** kernel. Sets measured before the CMs were saved
(everything before `0924_1`) need their CMs recovered from the raw data and the CSV $\rho$ the
first time; this takes a few minutes and is cached in `<set>_confusion_extracted.csv`.
Measured CMs (`<set>_confusion.csv`) are only ever read. Figures are written to `figures/`.

**Conventions.** Basis $|Q1\,Q2\rangle$ = 00, 01, 10, 11; Q1 is dataset `y0`. `CM[measured, prepared]`,
columns sum to 1. For this pure target the fidelity is linear in $\rho$:
$F = \langle\Psi^+|\rho|\Psi^+\rangle = \tfrac14\,(1 + \langle XX\rangle + \langle YY\rangle - \langle ZZ\rangle)$.

In [ ]:
%matplotlib inline
import os
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

import bell_analysis as ba
from snl_qblox.tomography_tools import compute_stokes, rearrange_stokes

FIGDIR = "figures"
os.makedirs(FIGDIR, exist_ok=True)

# Palette: first three categorical slots of the validated reference palette (all-pairs CVD-safe).
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, GRID, REF = "#0b0b0b", "#52514e", "#d4d4d8", "#8a8984"
DIVERGING = LinearSegmentedColormap.from_list("bell_div", ["#184f95", "#f0efec", "#a62f2e"])
SEQUENTIAL = LinearSegmentedColormap.from_list("bell_seq", ["#f0efec", "#86b6ef", "#1c5cab", "#0d366b"])

plt.rcParams.update({
    "font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9, "legend.fontsize": 8,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.5, "axes.axisbelow": True,
    "legend.frameon": False, "lines.linewidth": 1.5, "lines.markersize": 5,
    "savefig.dpi": 300, "savefig.bbox": "tight", "pdf.fonttype": 42,
})


def save(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(FIGDIR, f"{name}.{ext}"))


def day(label):
    return label.split("_")[0]

## 0. The data sets

The table lists every set found in this folder. *CM source* says whether the per-run confusion
matrices were saved by the measurement loop (`saved`) or recovered afterwards (`extracted`,
see section 6). The single-qubit pulse and $\sqrt{i\mathrm{SWAP}}$ lengths are read from each
set's instrument snapshot.

In [ ]:
sets = {}
for name in ba.list_sets():
    s = ba.BellSet(name)
    s.t_1q, s.t_2q = ba.snapshot_durations(s.tuids[0])
    sets[s.label] = s

print(f"{'set':8s}{'runs':>5s}  {'time':13s}{'tomo shots':>11s}{'CM shots':>9s}{'CM source':>11s}{'1Q ns':>7s}{'sqiSWAP ns':>11s}")
for k, s in sets.items():
    print(f"{k:8s}{len(s):>5d}  {s.times[0]:%H:%M}-{s.times[-1]:%H:%M}  {s.n_tomo:>9d}{s.n_cm:>9d}"
          f"{s.cm_source:>11s}{s.t_1q*1e9:>7.0f}{s.t_2q*1e9:>11.0f}")

## 1. From counts to a density matrix

**Sequence.** 16 ns buffer → $X_\pi$ on Q1 (100 ns) → 26 ns buffer → $\sqrt{i\mathrm{SWAP}}$
(~250 ns, SNAIL-pumped XY exchange) → virtual-Z correction that puts the Bell coherence on the
real axis → 16 ns buffer → one of 9 tomography pulse pairs (100 ns) → simultaneous readout.

**Tomography settings.** The 9 settings are two-qubit Cliffords whose decomposition is a
single-qubit pulse on each qubit, chosen from {I, $X_{90}$, $-Y_{90}$}. A $-Y_{90}$ pulse maps
$X\to Z$ and an $X_{90}$ maps $Y\to Z$, so after the pulse a Z-basis readout measures X, Y or Z
of each qubit. 3 × 3 = 9 settings give all 15 two-qubit Pauli expectation values (some
single-qubit ones are available from several settings; the reconstruction uses a fixed one).

**Step 1 – readout correction.** Each setting gives joint probabilities
$\mathbf p = (p_{00}, p_{01}, p_{10}, p_{11})$. The readout mixes them,
$\mathbf p_\text{meas} = \mathrm{CM}\,\mathbf p_\text{true}$, so the corrected probabilities are
$\mathbf q = \mathrm{CM}^{-1}\,\mathbf p_\text{meas}$.

**Step 2 – linear inversion.** Signed sums of $\mathbf q$ are Pauli expectation values, e.g.
$\langle XX\rangle = q_{00} - q_{01} - q_{10} + q_{11}$ in the setting where both qubits measure X.
The 15 values (the *Stokes vector*) give
$\rho_\text{lin} = \tfrac14\left(I\otimes I + \sum_P \langle P\rangle\,P\right)$.
Everything up to here is **linear** in the measured probabilities.

**Step 3 – maximum likelihood (MLE).** Shot noise makes $\rho_\text{lin}$ slightly unphysical
(small negative eigenvalues). `mle_residuals` finds the physical $\rho$ whose Stokes vector is
closest to the measured one. Because the Stokes vector is an orthogonal basis expansion, this is
the Frobenius-nearest density matrix, which has a closed form: keep the eigenvectors, subtract a
common $\mu$ from all eigenvalues, clip negatives to zero, with $\mu$ fixed by
$\mathrm{Tr}\,\rho = 1$. This is the **only nonlinear step**, and section 2 is about its
consequences.

In [ ]:
print("setting  Clifford   Q1 pulse  Q2 pulse   Q1 measures  Q2 measures")
meas = {"I": "Z", "X90": "Y", "mY90": "X"}
for k, idx in enumerate(ba.TOMO_SEQUENCE):
    a, b = ba._tomo_gates(idx)
    print(f"{k:>5d}    {idx:>5d}     {a:>6s}    {b:>6s}      {meas[a]:>5s}        {meas[b]:>5s}")

### Figure 1 – Pooled density matrix of the best set (0924_1): before and after the MLE

The 48 runs of `0924_1` are pooled (section 2): each run is corrected with its own saved CM and
linearly inverted, the 48 (unphysical) $\rho_{\rm lin}$ are averaged, and **one** MLE is applied to
the average. The three columns show that last step:

* **(a) pooled, before the MLE** – the average of the per-run $\rho_{\rm lin}$. Everything up to here is
  linear, so this is exactly one tomography with all the shots. It is not quite physical: its smallest
  eigenvalue is slightly negative.
* **(b) after the one MLE** – the nearest physical density matrix (the reported state and fidelity).
* **(c) the difference (b) − (a)**, on its own scale (×10⁻³). The MLE removes the negative eigenvalue
  by lowering all eigenvalues by the same amount, so the change is small and spread over the
  matrix; it is what separates the linear fidelity of (a) from the fidelity of (b).

Top row: real part; bottom row: imaginary part. The target has 0.5 on the four central elements
($|01\rangle\langle01|$, $|10\rangle\langle10|$ and the two coherences) and zero elsewhere. Colour is a
diverging blue–red scale with a neutral grey at zero, and every cell carries its value.

What to look for in (a)/(b): the populations of $|01\rangle$ and $|10\rangle$ are balanced (the
$\sqrt{i\mathrm{SWAP}}$ angle is right), the coherence is close to 0.5 (little dephasing), the
imaginary part is small (the Bell phase is close to zero), and $|00\rangle\langle00|$ is small
(little $T_1$ decay back to the ground state).

In [ ]:
best = sets["0924_1"]
rho_lin = best.lin.mean(0)            # (a) pooled, before the MLE (not physical)
rho = best.pooled_rho()               # (b) after the one MLE = mle(rho_lin)
F_best, F_best_se = best.pooled_F()
diff = rho - rho_lin                  # (c)
ev_lin, ev_mle = np.linalg.eigvalsh(rho_lin), np.linalg.eigvalsh(rho)
labels = ["00", "01", "10", "11"]
dmax = max(1.0, np.ceil(np.abs(diff).max() * 1e3))   # colour range of (c), in 1e-3


def draw(ax, M, vmax, fmt, cmap):
    im = ax.imshow(M, cmap=cmap, vmin=-vmax, vmax=vmax)
    for i in range(4):
        for j in range(4):
            v = M[i, j]
            txt = fmt.format(v)
            if float(txt) == 0:
                txt = "0"
            ax.text(j, i, txt, ha="center", va="center", fontsize=7,
                    color="white" if abs(v) > 0.6 * vmax else INK)
    ax.set_xticks(range(4), [f"|{l}⟩" for l in labels], fontsize=7)
    ax.set_yticks(range(4), [f"⟨{l}|" for l in labels], fontsize=7)
    ax.grid(False)
    for sp in ax.spines.values():
        sp.set_visible(False)
    return im


titles = [f"(a) pooled, before MLE\n$F_{{lin}}$ = {ba.fidelity(rho_lin):.4f},  min eig {ev_lin[0]:+.4f}",
          f"(b) after one MLE\n$F$ = {F_best:.4f} ± {F_best_se:.4f},  min eig 0",
          "(c) (b) − (a)\nin units of 10⁻³"]
fig, axes = plt.subplots(2, 3, figsize=(8.6, 5.4), constrained_layout=True)
for r, (part, name) in enumerate([(np.real, "Re"), (np.imag, "Im")]):
    im_ab = draw(axes[r, 0], part(rho_lin), 0.5, "{:+.3f}", DIVERGING)
    draw(axes[r, 1], part(rho), 0.5, "{:+.3f}", DIVERGING)
    im_c = draw(axes[r, 2], part(diff) * 1e3, dmax, "{:+.1f}", DIVERGING)
    axes[r, 0].set_ylabel(f"{name} $\\rho$", fontsize=9)
for c, t in enumerate(titles):
    axes[0, c].set_title(t, loc="left", fontsize=8)
cb1 = fig.colorbar(im_ab, ax=axes[:, :2], shrink=0.6, ticks=[-0.5, 0, 0.5], location="left", pad=0.08)
cb1.outline.set_visible(False)
cb2 = fig.colorbar(im_c, ax=axes[:, 2], shrink=0.6, ticks=[-dmax, 0, dmax])
cb2.outline.set_visible(False)
cb2.set_label("×10⁻³")
save(fig, "fig1_pooled_rho_0924_1")
plt.show()

print(f"eigenvalues before MLE: {np.round(ev_lin, 4)}")
print(f"eigenvalues after MLE:  {np.round(ev_mle, 4)}")
print(f"F: linear (before) {ba.fidelity(rho_lin):.4f} -> after one MLE {F_best:.4f} ± {F_best_se:.4f};"
      f"  max |difference| {np.abs(diff).max():.4f}")

### Figure 2 – Pauli expectation values

The same pooled state as 15 Pauli expectation values, in the order used in `SNL315_plot.ipynb`
(single-qubit terms of Q2, of Q1, then the two-qubit correlators). In each tick label the top letter is the Q2 operator and the bottom one the Q1 operator. The target (grey outline) has
only $\langle XX\rangle = \langle YY\rangle = +1$ and $\langle ZZ\rangle = -1$; those three bars
alone set the fidelity, $F = (1 + \langle XX\rangle + \langle YY\rangle - \langle ZZ\rangle)/4$.
Small non-zero single-qubit or cross terms indicate coherent errors (a residual $|00\rangle$
admixture from an imperfect $X_\pi$ shows up in $\langle IX\rangle$, $\langle XI\rangle$,
$\langle XZ\rangle$, $\langle ZX\rangle$; section 5).

In [ ]:
pauli_labels = ["Z\nI", "X\nI", "Y\nI", "I\nZ", "I\nX", "I\nY", "Z\nZ", "Z\nX", "Z\nY",
                "X\nZ", "X\nX", "X\nY", "Y\nZ", "Y\nX", "Y\nY"]
meas_s = rearrange_stokes(compute_stokes(rho))
ideal_s = rearrange_stokes(compute_stokes(ba.IDEAL))
x = np.arange(15)

fig, ax = plt.subplots(figsize=(6.8, 2.6))
ax.axvspan(-0.5, 2.5, color=GRID, alpha=0.25, lw=0); ax.axvspan(2.5, 5.5, color=GRID, alpha=0.5, lw=0)
ax.bar(x, ideal_s, 0.78, facecolor="none", edgecolor=REF, lw=1.2, label="target", zorder=1)
ax.bar(x, meas_s, 0.56, color=BLUE, label="measured (pooled)", zorder=2)
ax.axhline(0, color=INK2, lw=0.8)
for i in (6, 10, 14):
    ax.text(i, meas_s[i] + (0.06 if meas_s[i] > 0 else -0.06), f"{meas_s[i]:+.3f}", ha="center",
            va="bottom" if meas_s[i] > 0 else "top", fontsize=7.5, color=INK)
ax.set_xticks(x, pauli_labels); ax.set_ylim(-1.2, 1.2); ax.set_ylabel("expectation value")
ax.text(1, -1.1, "Q2", ha="center", color=INK2); ax.text(4, -1.1, "Q1", ha="center", color=INK2)
ax.text(10, -1.1, "Q1 Q2", ha="center", color=INK2)
ax.legend(loc="upper left", ncol=2)
ax.grid(axis="x", visible=False)
save(fig, "fig2_pauli_0924_1")
plt.show()

## 2. Why the runs must be pooled before the MLE

The CSV files store each run's **MLE** density matrix, and the fidelity used to be quoted as
the average of those per-run fidelities. That average is biased low, and the reason is the MLE.

**The mechanism.** A single run's $\rho_\text{lin}$ has noise of order $1/\sqrt{\text{shots}}$ in
every element. For a nearly pure state three eigenvalues sit near zero, so noise pushes one of
them below zero almost every time. The MLE removes it by lowering all eigenvalues by the same
$\mu$ – including the large one that carries the Bell state – and the fidelity drops. Noise
can make a small eigenvalue negative but the projection only ever removes weight from the Bell
eigenvalue, so this loss has one sign and **does not average out** across runs.

**Pooling.** Up to the MLE every step is linear, so averaging the runs' $\rho_\text{lin}$ is
exactly the same as one tomography with all $N\times$shots. The noise shrinks by $\sqrt N$
*before* the single MLE, which then only removes what is left – mostly systematic
(tomography-pulse errors and an imperfect readout model), not noise.

**Figure 3.** (a) The most negative eigenvalue of each run's $\rho_\text{lin}$ (orange) against the
pooled one (blue line). (b) One run before and after the MLE: the negative eigenvalue is set to
zero and the others, including the Bell eigenvalue, drop by $\mu$. (c) Per-run fidelity before
(aqua) and after (orange) the MLE; the linear values scatter symmetrically around the pooled
linear fidelity, the MLE values are shifted down as a block.

In [ ]:
L = best.lin
ev = np.array([np.linalg.eigvalsh(r) for r in L])
ev_pooled = np.linalg.eigvalsh(L.mean(0))
F_lin, F_mle = best.per_run_F_linear(), best.per_run_F()
w0 = ev[0]
from snl_qblox.tomography_tools import _project_eigs_to_simplex
w0p = _project_eigs_to_simplex(w0, 1.0)

fig, axes = plt.subplots(1, 3, figsize=(7.6, 2.9), layout="constrained")
ax = axes[0]
ax.hist(ev[:, 0], bins=14, color=ORANGE, edgecolor="white", linewidth=1)
ax.axvline(ev_pooled[0], color=BLUE, lw=2)
ax.text(ev_pooled[0], ax.get_ylim()[1] * 0.92, " pooled", color=INK, va="top")
ax.set_xlabel("most negative eigenvalue of $\\rho_{lin}$"); ax.set_ylabel("runs"); ax.set_title("(a) per run vs pooled", loc="left")

ax = axes[1]
xx = np.arange(3)
ax.bar(xx - 0.19, w0[:3], 0.36, color=AQUA, label="before MLE")
ax.bar(xx + 0.19, w0p[:3], 0.36, color=ORANGE, label="after MLE")
ax.axhline(0, color=INK2, lw=0.8)
ax.set_xticks(xx, ["$\\lambda_1$", "$\\lambda_2$", "$\\lambda_3$"])
ax.set_title("(b) one run, before/after MLE", loc="left"); ax.legend(loc="upper left")
ax.text(0.98, 0.04, f"Bell eigenvalue\n{w0[3]:.4f} → {w0p[3]:.4f}", transform=ax.transAxes, ha="right", va="bottom", color=INK)

ax = axes[2]
bins = np.linspace(min(F_mle.min(), F_lin.min()) - 0.002, max(F_mle.max(), F_lin.max()) + 0.002, 18)
ax.hist(F_lin, bins=bins, color=AQUA, alpha=0.85, edgecolor="white", linewidth=1, label="per run, linear")
ax.hist(F_mle, bins=bins, color=ORANGE, alpha=0.85, edgecolor="white", linewidth=1, label="per run, MLE")
ax.axvline(F_best, color=BLUE, lw=2, label="pooled, one MLE")
ax.set_xlabel("fidelity"); ax.set_title("(c) fidelity", loc="left")
fig.legend(*ax.get_legend_handles_labels(), loc="outside lower right", ncol=3, fontsize=7.5)
save(fig, "fig3_pooling_mechanism")
plt.show()

print(f"per-run most negative eigenvalue: mean {ev[:,0].mean():+.4f} (range {ev[:,0].min():+.4f} to {ev[:,0].max():+.4f}); pooled {ev_pooled[0]:+.4f}")
print(f"mean per-run F: linear {F_lin.mean():.4f}, after MLE {F_mle.mean():.4f}  (MLE costs {F_lin.mean()-F_mle.mean():.4f})")
print(f"pooled: linear {ba.fidelity(L.mean(0)):.4f}, after one MLE {F_best:.4f}  (MLE costs {ba.fidelity(L.mean(0))-F_best:.4f})")

## 3. Which estimator is right? A simulation with a known answer

The data alone cannot say which number is *correct*, so the estimators are tested on synthetic
data whose true fidelity is known. The truth is the pooled 0924_1 state and its mean CM. Each
simulated set has 48 runs; every run draws multinomial tomography counts and a multinomial CM
(the notebook measures a fresh CM per run), and the three estimators are applied:

* **per-run MLE, averaged** – the original notebook method;
* **per-run linear, averaged** – no MLE at all (unbiased in expectation, but individual values can exceed 1);
* **pooled, one MLE** – the recommended method.

This is repeated for the three shot settings that were actually used. **Figure 4** shows the
estimator minus the true fidelity (mean ± sd over repetitions). The per-run MLE bias is large
and grows as the shot count drops; the pooled estimator is unbiased to < 0.1 %.

In [ ]:
rho_true, CM_true = best.pooled_rho(), best.CM.mean(0)
F_true = ba.fidelity(rho_true)
configs = [(1000, 500), (2000, 2000), (4000, 2000)]
sim = {c: ba.simulate_estimators(rho_true, CM_true, n_runs=48, n_tomo=c[0], n_cm=c[1], reps=100) for c in configs}

colors = {"per-run MLE, averaged": ORANGE, "per-run linear, averaged": AQUA, "pooled, one MLE": BLUE}
markers = {"per-run MLE, averaged": "v", "per-run linear, averaged": "s", "pooled, one MLE": "o"}
fig, ax = plt.subplots(figsize=(5.2, 2.8))
for i, c in enumerate(configs):
    for j, (k, col) in enumerate(colors.items()):
        v = (sim[c][k] - F_true) * 100
        ax.errorbar(i + (j - 1) * 0.18, v.mean(), yerr=v.std(), fmt=markers[k], color=col, ms=7,
                    capsize=0, elinewidth=2, label=k if i == 0 else None)
ax.axhline(0, color=INK2, lw=0.8)
ax.set_xticks(range(3), [f"{a} / {b}" for a, b in configs])
ax.set_xlabel("shots per run: tomography / confusion matrix")
ax.set_ylabel("estimate − true F (%)"); ax.legend(loc="lower right")
ax.grid(axis="x", visible=False)
save(fig, "fig4_estimator_bias")
plt.show()

print(f"true F = {F_true:.4f}")
for c in configs:
    print(f"{c}: " + ", ".join(f"{k}: {np.mean(v)-F_true:+.4f}±{np.std(v):.4f}" for k, v in sim[c].items()))

## 4. Fidelity of every set

**Figure 5.** For each set: every run's MLE fidelity (grey dots), their average (orange triangle –
the old estimator) and the pooled fidelity with its bootstrap error bar (blue circle). The
bootstrap resamples whole runs with replacement and redoes the pooling, so it includes any real
run-to-run fluctuation; it is the uncertainty of the mean, not the scatter of a single run.

How to read the trend:

* The gap between orange and blue is the MLE bias of section 3. It is ~1.5 % for the 1000/500-shot
  sets and shrinks with more shots.
* **July (0723/0724)** pooled values are *inflated*: those CMs were measured with an $X_\pi$ that was
  ~4 % off (section 5), and the CM absorbs that error, so the readout correction also "corrects"
  part of the gate error. They are not comparable with the September sets.
* **0923 → 0924_1**: better calibration and more shots; 0924_1 reaches the $T_1/T_2$ limit (section 5).

In [ ]:
order = list(sets)
pooled = {k: sets[k].pooled_F() for k in order}
perrun = {k: sets[k].per_run_F() for k in order}

fig, ax = plt.subplots(figsize=(7.0, 3.0))
rng = np.random.default_rng(3)
for i, k in enumerate(order):
    ax.scatter(i + rng.uniform(-0.18, 0.18, len(perrun[k])), perrun[k], s=6, color=REF, alpha=0.5, lw=0,
               label="single runs (MLE)" if i == 0 else None)
    ax.plot(i - 0.05, perrun[k].mean(), "v", color=ORANGE, ms=7, label="average of per-run F" if i == 0 else None)
    ax.errorbar(i + 0.05, pooled[k][0], yerr=pooled[k][1], fmt="o", color=BLUE, ms=6, elinewidth=2,
                label="pooled, one MLE" if i == 0 else None)
prev = None
for i, k in enumerate(order):
    if prev is not None and day(k) != prev:
        ax.axvline(i - 0.5, color=GRID, lw=1)
    prev = day(k)
ax.set_xticks(range(len(order)), order, rotation=0)
ax.set_ylabel("Bell-state fidelity"); ax.set_ylim(0.93, 1.0)
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=3)
ax.grid(axis="x", visible=False)
save(fig, "fig5_fidelity_all_sets")
plt.show()

print(f"{'set':8s}{'runs':>5s}{'per-run mean':>14s}{'per-run sd':>12s}{'pooled F':>10s}{'± boot':>8s}")
for k in order:
    print(f"{k:8s}{len(sets[k]):>5d}{perrun[k].mean():>14.4f}{perrun[k].std():>12.4f}{pooled[k][0]:>10.4f}{pooled[k][1]:>8.4f}")

## 5. Gate calibration and the error budget

A Lindblad model of the whole sequence (`ba.ErrorModel`) is fitted to each set's **raw** joint
probabilities, which do not depend on any confusion matrix. Its free parameters are the
per-qubit readout fidelities, the $X_\pi$ amplitude error, the $\sqrt{i\mathrm{SWAP}}$ angle error,
a Z phase on each qubit (their difference is the Bell phase) and the tomography-pulse amplitude
errors; $T_1$ and $T_2$ are fixed. Thermal population is fixed at zero because it cannot be
separated from readout error with this data.

*Coherence times used.* For the 0924 sets the values measured next to them (16:32 for
`0924_0`, 17:41 for `0924_1`, 18:50 for `0924_2`, section 8) with $T_2 = T_2^{\rm echo}$ and low TLS dips in the
repeats removed (`ba.tls_dips`, section 8); all older sets
use 40 µs for both qubits (not measured alongside those sets). In the 18:50 run the first 8
repeats of Q2 fall in a transient dip (T1 ≈ 23 µs instead of ≈ 31 µs, Figure 11); they are
excluded, so the budget describes Q2's normal coherence rather than a few-minute excursion.
`0924_3` (19:07–19:32) has no T1/T2 measurement of its own and uses the same 18:50 values.

**Figure 6** tracks the calibration errors over the sets. (a) The fitted $X_\pi$ error. On the
July sets it is ~−3.7 %, stable across four sets and a recalibration, and disappears after the
September calibrations (which added Ramsey and ALLXY). The fit cannot tell an amplitude error
from a ~300 kHz detuning of Q1 – both leave the same coherent $|00\rangle$ admixture – so read it
as "amplitude-equivalent". (b) The model-free signature of that admixture: the magnitude of the
$\rho_{00,01}$ coherence in the pooled state. (c) The $\sqrt{i\mathrm{SWAP}}$ angle error, within
~1 % everywhere.

In [ ]:
T1T2_1850 = "20260924-185043-868-5463c8"
DIP_EXCLUDE = {"Q2": range(8)}
T12 = {"0924_0": ba.coherence_times("20260924-163223-748-911d26"),
       "0924_1": ba.coherence_times("20260924-174141-671-4869f0"),
       # Q2 repeats 0-7 of the 18:50 run sit in a transient dip (section 8, Figure 11); excluded.
       "0924_2": ba.coherence_times(T1T2_1850, exclude=DIP_EXCLUDE)}
T12["0924_3"] = T12["0924_2"]   # no T1/T2 after 0924_3; nearest run is 18:50 (17 min before)

def t1t2_for(label, t2_kind="T2echo"):
    if label in T12:
        c = T12[label]
        return ([c["Q1"]["T1"][0] * 1e-6, c["Q2"]["T1"][0] * 1e-6],
                [c["Q1"][t2_kind][0] * 1e-6, c["Q2"][t2_kind][0] * 1e-6])
    return [40e-6, 40e-6], [40e-6, 40e-6]

fits = {}
for k in order:
    T1, T2 = t1t2_for(k)
    fits[k] = ba.fit_error_model(sets[k], T1, T2)
coh0001 = {k: abs(sets[k].pooled_rho()[0, 1]) for k in order}

P = ba.PARAMS
fig, axes = plt.subplots(3, 1, figsize=(6.4, 5.2), sharex=True, constrained_layout=True)
xs = np.arange(len(order))
ex = np.array([fits[k][1][P.index("eps_Xpi")] for k in order]) * 100
ex_err = np.array([fits[k][2][P.index("eps_Xpi")] for k in order]) * 100
sw = np.array([fits[k][1][P.index("eps_swap")] for k in order]) * 100
sw_err = np.array([fits[k][2][P.index("eps_swap")] for k in order]) * 100
for ax, y, e, lab in [(axes[0], ex, ex_err, "(a) $X_\\pi$ error (%)"), (axes[2], sw, sw_err, "(c) $\\sqrt{iSWAP}$ angle error (%)")]:
    ax.axhline(0, color=INK2, lw=0.8)
    ax.errorbar(xs, y, yerr=np.minimum(e, 5), fmt="o", color=BLUE, ms=6, elinewidth=2)
    ax.set_ylabel(lab, fontsize=8)
axes[1].plot(xs, [coh0001[k] for k in order], "o", color=BLUE, ms=6)
axes[1].set_ylabel("(b) $|\\rho_{00,01}|$", fontsize=8); axes[1].set_ylim(0, None)
for ax in axes:
    ax.grid(axis="x", visible=False)
axes[-1].set_xticks(xs, order)
save(fig, "fig6_gate_calibration")
plt.show()

print(f"{'set':8s}{'chi2/dof':>9s}{'X_pi %':>14s}{'swap %':>14s}{'arg rho01,10 deg':>18s}{'tomo Q1/Q2 %':>16s}")
for k in order:
    m, p, e, chi = fits[k]
    print(f"{k:8s}{chi:>9.1f}{p[6]*100:>8.2f}±{e[6]*100:<5.2f}{p[7]*100:>8.2f}±{e[7]*100:<5.2f}"
          f"{np.degrees(p[9]-p[8]):>18.2f}{p[10]*100:>9.2f}/{p[11]*100:<6.2f}")

### Figure 7 – Error budget of 0924_1 and 0924_2

Starting from a perfect Bell state, each row adds one effect and shows the resulting fidelity:

1. **$T_1/T_2$ only** – decay during the ~0.4 µs from the start of the $X_\pi$ to the end of the
   $\sqrt{i\mathrm{SWAP}}$, with the measured coherence times.
2. **+ gate errors** – the fitted $X_\pi$, $\sqrt{i\mathrm{SWAP}}$ and Bell-phase errors (each costs
   < 0.05 % here). This is the model's **true state fidelity**.
3. **Reconstructed, pure-readout correction** – what tomography reports if the readout correction
   removed *only* readout error: decay during the 116 ns tomography segment and the
   tomography-pulse errors are then charged to the state. A lower bound.
4. **Reconstructed, notebook-style CM** – the model also simulates how the loop measures its CM
   (prep $X_\pi$ + 116 ns padded identity + readout). That CM contains the same pre-readout decay
   as the tomography, so correcting with it removes that decay as well. This is the like-for-like
   prediction of the measured pooled number.

The measured pooled fidelity (orange) sits on the notebook-style prediction for both sets: the
result is fully explained by $T_1/T_2$ plus small gate errors. For 0924_2 the coherence times
come from the 18:50 run with Q2's transient dip excluded (Figure 11). $T_2$ is the echo value throughout.

In [ ]:
def budget_rows(label):
    m, p, e, chi = fits[label]
    bud = ba.error_budget(m, p)
    rows = [("perfect state", 1.0),
            ("$T_1/T_2$ only (state)", bud["F_T1T2"]),
            ("+ gate errors (state)", bud["F_state"]),
            ("reconstructed, pure-readout CM", bud["F_recon_pure_readout"]),
            ("reconstructed, notebook-style CM", bud["F_recon_notebook_cm"])]
    return rows, bud

budget_sets = ["0924_1", "0924_2"]
fig, axes = plt.subplots(1, 2, figsize=(9.0, 2.9), sharey=True, constrained_layout=True)
for ax, label in zip(axes, budget_sets):
    rows, bud = budget_rows(label)
    Fm, Fm_se = pooled[label]
    y = np.arange(len(rows))[::-1]
    ax.hlines(y, 0.98, [v for _, v in rows], color=GRID, lw=1)
    ax.plot([v for _, v in rows], y, "o", color=BLUE, ms=7, label="model")
    ax.errorbar(Fm, y[-1], xerr=Fm_se, fmt="D", color=ORANGE, ms=7, lw=2, capsize=0, label="measured pooled")
    for yy, (lab, v) in zip(y, rows):
        ax.text(0.9805, yy + 0.2, f"{lab}:  {v:.4f}", fontsize=7.5, color=INK, va="bottom")
    ax.set_title(f"{label}   (measured {Fm:.4f} ± {Fm_se:.4f})", loc="left")
    ax.set_yticks([]); ax.set_xlim(0.98, 1.0015); ax.set_ylim(-0.5, len(rows) - 0.3)
    ax.set_xlabel("Bell-state fidelity"); ax.spines["left"].set_visible(False); ax.grid(axis="y", visible=False)
    print(f"{label}: " + ", ".join(f"{k} {v:.4f}" for k, v in bud.items() if not isinstance(v, dict))
          + ";  gate-error costs " + ", ".join(f"{k} {v:+.5f}" for k, v in bud["costs"].items()))
axes[-1].legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=7.5)
save(fig, "fig7_error_budget_0924")
plt.show()

## 6. Readout and the confusion matrix

**What the CM contains.** `construct_confusion_matrix` prepares $|00\rangle, |01\rangle, |10\rangle,
|11\rangle$ with $X_\pi$ pulses inside the same `_tomo_build` schedule, and reads the populations
of the no-pulse setting. Each column is therefore *readout error × everything that happens
between the prep pulse and the readout*: $T_1$ decay during the $X_\pi$ and the 116 ns padded
identity, and any $X_\pi$ error. This is why (i) its diagonal is below the product of the
single-qubit readout fidelities, (ii) a CM correction lifts the fidelity to roughly the
"before tomography" state fidelity (Figure 7), and (iii) a miscalibrated $X_\pi$ (July) leaks
into the corrected result.

**Recovering unsaved CMs.** For sets measured before the CMs were saved, each run's CM was
reconstructed from the raw counts and the CSV $\rho$: the linear-inversion Stokes vector is
linear in $\mathrm{CM}^{-1}$, so the CM follows from a constrained fit through the same MLE.
Tomography leaves a few combinations of CM entries invisible, and the MLE hides more, so the CM
is additionally required to be physical (entries in [0, 1], columns summing to 1).

**Figure 8** validates this on 0924_1, where the CMs *were* saved. (a) The mean saved CM.
(b) Extracted versus saved entries, every run: they fall on the identity line to ~0.003
r.m.s.; the remaining differences lie in the direction tomography cannot see, so any quantity
computed from $\rho$ is identical with either CM. (c) Per-qubit readout fidelities (from the CM
marginals) across all sets.

In [ ]:
CM_saved = best.CM
CM_ext = ba.extracted_cms(best.name, best.tuids, best.P, best.rho_csv)
diff = CM_ext - CM_saved
F_saved = np.array([ba.fidelity(ba.mle(ba.lin_rho(p_, c))) for p_, c in zip(best.P, CM_saved)])
F_ext = np.array([ba.fidelity(ba.mle(ba.lin_rho(p_, c))) for p_, c in zip(best.P, CM_ext)])

fig, axes = plt.subplots(1, 3, figsize=(7.4, 2.6), constrained_layout=True, gridspec_kw={"width_ratios": [1, 1, 1.25]})
ax = axes[0]
Cm = CM_saved.mean(0)
ax.imshow(Cm, cmap=SEQUENTIAL, vmin=0, vmax=1)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{Cm[i, j]:.3f}", ha="center", va="center", fontsize=7, color="white" if Cm[i, j] > 0.5 else INK)
ax.set_xticks(range(4), labels); ax.set_yticks(range(4), labels)
ax.set_xlabel("prepared"); ax.set_ylabel("measured"); ax.set_title("(a) mean saved CM, 0924_1", loc="left"); ax.grid(False)
for sp in ax.spines.values(): sp.set_visible(False)

ax = axes[1]
diag = np.eye(4, dtype=bool)
ax.plot([0, 1], [0, 1], color=REF, lw=1)
ax.scatter(CM_saved[:, diag].ravel(), CM_ext[:, diag].ravel(), s=8, color=BLUE, lw=0, label="diagonal")
ax.scatter(CM_saved[:, ~diag].ravel(), CM_ext[:, ~diag].ravel(), s=8, color=ORANGE, lw=0, label="off-diagonal")
ax.set_xlabel("saved"); ax.set_ylabel("extracted"); ax.set_title("(b) extracted vs saved", loc="left")
ax.set_aspect("equal"); ax.legend(loc="upper left", fontsize=7)

ax = axes[2]
marg = {k: [np.mean([ba.bce.marginal_fidelities(c)[q] for c in sets[k].CM]) for q in ["Fg_Q1", "Fg_Q2", "Fe_Q1", "Fe_Q2"]] for k in order}
arr = np.array([marg[k] for k in order])
ax.plot(xs, arr[:, 0], "o-", color=BLUE, ms=4, lw=1, label="Q1")
ax.plot(xs, arr[:, 1], "s-", color=ORANGE, ms=4, lw=1, label="Q2")
ax.plot(xs, arr[:, 2], "o--", color=BLUE, ms=4, lw=1)
ax.plot(xs, arr[:, 3], "s--", color=ORANGE, ms=4, lw=1)
ax.text(xs[-1] + 0.3, arr[-1, :2].mean(), "$F_g$", va="center", color=INK)
ax.text(xs[-1] + 0.3, arr[-1, 2:].mean(), "$F_e$", va="center", color=INK)
ax.set_xticks(xs, order, rotation=90, fontsize=7); ax.set_ylabel("assignment fidelity")
ax.set_title("(c) readout per set (solid $F_g$, dashed $F_e$)", loc="left", fontsize=8); ax.legend(loc="center left", fontsize=7)
ax.grid(axis="x", visible=False)
save(fig, "fig8_confusion_matrix")
plt.show()

print(f"extracted - saved: rms {np.sqrt((diff**2).mean()):.4f}, max {np.abs(diff).max():.4f}")
print(f"per-run F with extracted vs saved CM: max |diff| {np.abs(F_ext - F_saved).max():.1e}")

### Cross-check: does a set's CM describe that set's readout? (0924_1 vs 0924_3)

`0924_3` was measured right after a multiplexed readout calibration (19:02) and its pooled
fidelity came out ~0.35 % higher than 0924_1/0924_2 – above the $T_1/T_2$ limit of the model,
which is the same warning sign as the July sets. Two checks separate the state from the
correction:

* **Raw data (no CM at all).** 0924_1 and 0924_3 have the same readout (the model fitted to raw
  counts gives Q1 $F_e$ 0.8755 vs 0.8750, Q2 $F_e$ 0.907 vs 0.906), so their *uncorrected* Pauli
  correlators can be compared directly. They agree within error: the state did not improve.
* **Swap the CMs.** Each set is corrected with the other set's mean measured CM. If the data
  differed, the rows would differ; if the CMs differ, the columns do.

The table shows the column pattern: 0924_3's CMs inflate *either* data set by ~0.4 %, and
0924_3's data corrected with 0924_1's CMs give ≈ 0.991, in line with the other sets. The 0924_3
CMs show Q1-excited columns ($|10\rangle$, $|11\rangle$) reading Q1 as ground ~0.5 % more often
(≈ 3σ), while the Bell data corrected with them show no such drop – Q1 is read as excited
slightly *more* often. The CM, measured with prepared basis states and no pump pulse, did not
describe the readout of the Bell runs it corrected, so 0924_3's corrected fidelity overstates
the state. Its raw-data-consistent value is the one corrected with 0924_1's CM.

In [ ]:
def pooled_with(s, CM, n_boot=300, seed=0):
    L = np.array([ba.lin_rho(p_, CM) for p_ in s.P])
    rng = np.random.default_rng(seed)
    n = len(L)
    boot = [ba.fidelity(ba.mle(L[rng.integers(0, n, n)].mean(0))) for _ in range(n_boot)]
    return ba.fidelity(ba.mle(L.mean(0))), float(np.std(boot))

pair = ["0924_1", "0924_3"]
print("pooled F; rows = data, columns = mean measured CM used")
print(f"{'':12s}" + "".join(f"{'CM ' + k:>20s}" for k in pair))
for kd in pair:
    print(f"data {kd:7s}" + "".join(f"{pooled_with(sets[kd], sets[kc].CM.mean(0))[0]:>13.4f}±{pooled_with(sets[kd], sets[kc].CM.mean(0))[1]:.4f}" for kc in pair))

print("\nraw (uncorrected) correlators:")
for k in pair:
    P_ = sets[k].P
    sr = ba.bce.stokes_lin(np.eye(4), P_.mean(0))
    se = np.std([ba.bce.stokes_lin(np.eye(4), x) for x in P_], 0) / np.sqrt(len(P_))
    print(f"  {k}: <XX> {sr[4]:.4f}±{se[4]:.4f}  <YY> {sr[9]:.4f}±{se[9]:.4f}  <ZZ> {sr[14]:.4f}±{se[14]:.4f}"
          f"  raw (1+XX+YY-ZZ)/4 = {(1 + sr[4] + sr[9] - sr[14]) / 4:.4f}")
d = sets["0924_3"].CM.mean(0) - sets["0924_1"].CM.mean(0)
se = np.sqrt(sets["0924_3"].CM.var(0) / len(sets["0924_3"]) + sets["0924_1"].CM.var(0) / len(sets["0924_1"]))
print("\nmean CM 0924_3 - 0924_1, in units of its standard error:")
print(np.round(d / np.where(se > 0, se, np.nan), 1))

## 7. Run-to-run stability (0924_1)

Each run's fidelity, Bell phase ($\arg\rho_{01,10}$), $|10\rangle$–$|01\rangle$ population
imbalance (the $\sqrt{i\mathrm{SWAP}}$ angle) and readout fidelities are plotted against time.
The shaded band is the mean ± the sd expected from **shot noise alone**, obtained by resampling
4000 tomography shots and a 2000-shot CM around the set's average. Points staying inside the
band mean the scatter is statistical; a ratio sd/shot-noise well above 1 means a real
fluctuation. In 0924_1 the fidelity, coherence and imbalance scatter at the shot-noise level;
the Bell phase and readout fidelities show a small real jitter (ratio 1.2–1.5) that does not
reach the fidelity, because each run is corrected with its own CM.

In [ ]:
obs = best.observables()
sn = best.shot_noise_sd()
import matplotlib.dates as mdates
tmin = np.array(best.times)          # clock time of each run
panels = [("F", "fidelity (per run)", 1), ("phase_deg", "Bell phase (deg)", 1), ("imbalance", "$p_{10}-p_{01}$", 1)]

fig, axes = plt.subplots(4, 1, figsize=(6.4, 6.0), sharex=True, constrained_layout=True)
for ax, (k, lab, _) in zip(axes, panels):
    mu = obs[k].mean()
    ax.fill_between([tmin.min(), tmin.max()], mu - sn[k], mu + sn[k], color=BLUE, alpha=0.15, lw=0)
    ax.plot(tmin, obs[k], "o-", color=BLUE, ms=3, lw=0.7)
    ax.set_ylabel(lab, fontsize=8)
    ax.text(1.0, 1.0, f"sd/shot-noise = {obs[k].std()/sn[k]:.1f}", transform=ax.transAxes, ha="right", va="bottom", fontsize=7.5, color=INK2)
ax = axes[3]
for k, col, mk, lab in [("Fe_Q1", BLUE, "o", "Q1"), ("Fe_Q2", ORANGE, "s", "Q2")]:
    mu = obs[k].mean()
    ax.fill_between([tmin.min(), tmin.max()], mu - sn[k], mu + sn[k], color=col, alpha=0.15, lw=0)
    ax.plot(tmin, obs[k], mk + "-", color=col, ms=3, lw=0.7, label=f"{lab}  (sd/shot-noise {obs[k].std()/sn[k]:.1f})")
ax.set_ylabel("$F_e$ from CM", fontsize=8); ax.legend(loc="lower right", fontsize=7, ncol=2)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
axes[-1].set_xlabel(f"time ({best.times[0]:%Y-%m-%d})")
save(fig, "fig9_stability_0924_1")
plt.show()

print(f"{'quantity':12s}{'mean':>9s}{'sd':>9s}{'shot sd':>9s}{'ratio':>7s}")
for k in obs:
    print(f"{k:12s}{obs[k].mean():>9.4f}{obs[k].std():>9.4f}{sn[k]:>9.4f}{obs[k].std()/sn[k]:>7.1f}")

## 8. Coherence times

Three simultaneous $T_1$ / $T_2$ measurements bracket the 0924 Bell sets (fitted with
`helpers.t1_and_t2_fit`; failed repeats are skipped; bars are standard errors). Low TLS dips
in the $T_1$ and $T_2^{\rm echo}$ repeats are removed before averaging: `ba.tls_dips` flags the
$k$ lowest points ($k < n/2$) when every one lies more than 3σ below the mean of the rest; the
removed values are printed below. Ramsey $T_2^*$ is averaged over all repeats: its low repeats
come from the qubit frequency jumping, which is real dephasing, not a TLS event. The error model
uses $T_1$ and $T_2^{\rm echo}$; $T_2^*$ is shown for reference.

Q1 is stable ($T_1$ ≈ 56–66 µs, $T_2^{\rm echo}$ ≈ 44–49 µs). Q2 degrades through the evening: its
Ramsey $T_2^*$ drops from 40 to 27 µs between 16:32 and 17:41 (echo 48 → 41 µs, so most of it is
slow frequency noise that the echo refocuses), and by 18:50 its $T_1$ has also dropped
(37 → 31 µs, dip excluded; Figure 11) – low-frequency noise or a fluctuator near Q2. Because the
Bell sequence is only ~0.6 µs long, this costs ≤ 0.1 % per step, which is why 0924_1 and 0924_2
have the same fidelity within error. The remaining ~0.7 % infidelity is essentially all
$T_1/T_2$, with Q2 the largest contributor.

In [ ]:
qty = ["T1", "T2echo", "T2*"]
fig, axes = plt.subplots(1, 2, figsize=(6.4, 2.6), sharey=True, constrained_layout=True)
for ax, q in zip(axes, ["Q1", "Q2"]):
    for j, (lab, col, mk) in enumerate([("0924_0", BLUE, "o"), ("0924_1", ORANGE, "s"), ("0924_2", AQUA, "D")]):
        v = [T12[lab][q][t] for t in qty]
        ax.errorbar(np.arange(3) + (j - 1) * 0.22, [a for a, _ in v], yerr=[b for _, b in v], fmt=mk, color=col, ms=6,
                    elinewidth=2, label={"0924_0": "16:32 (with 0924_0)", "0924_1": "17:41 (before 0924_1)",
                                         "0924_2": "18:50 (after 0924_2, Q2 dip excluded)"}[lab])
    ax.set_xticks(range(3), ["$T_1$", "$T_2^{echo}$", "$T_2^*$"]); ax.set_title(q, loc="left")
    ax.grid(axis="x", visible=False)
axes[0].set_ylabel("µs"); axes[0].set_ylim(0, None); axes[1].legend(loc="lower left", fontsize=7)
save(fig, "fig10_coherence_times")
plt.show()

for lab, c in T12.items():
    for q in ["Q1", "Q2"]:
        print(lab, q, {t: f"{c[q][t][0]:.1f}±{c[q][t][1]:.1f}" for t in qty})

print("\nlow TLS dips removed automatically (ba.tls_dips), in us:")
for lab, tuid, exc in [("16:32", "20260924-163223-748-911d26", None),
                       ("17:41", "20260924-174141-671-4869f0", None),
                       ("18:50", T1T2_1850, DIP_EXCLUDE)]:
    d = ba.tls_dropped(tuid, exclude=exc)
    print(lab, {q: {k: np.round(v, 1).tolist() for k, v in d[q].items() if len(v)} for q in d})

### Figure 11 – Q2's transient dip at 18:50

The 18:50 measurement has 20 interleaved repeats of T1, echo and Ramsey (T1, echo, Ramsey, then
the next repeat). No per-repeat timestamps are stored, but the whole run took ≤ 10 min, so
each repeat is ≤ 30 s. Plotted per repeat, Q2's T1 is 16–25 µs for the first 8 repeats and
≈ 31 µs afterwards, with the echo and Ramsey times depressed over the same stretch, while Q1 is
steady throughout. T1 and T2 dropping together for a few minutes and then recovering is the
typical signature of a two-level fluctuator (TLS) passing near Q2's frequency. The shaded
repeats are the ones excluded from the error budget (`DIP_EXCLUDE`); the dashed lines are Q2's
mean with (grey) and without (orange) them.

Two consequences: the average over all repeats (T1 28.4 µs, $T_2^*$ 25.9 µs) mixes two regimes
and understates Q2's normal coherence; and during such a dip the Bell-state fidelity drops by
only ≈ 0.14 % (model), below the resolution of a 12-run block of Bell data, so dips like this
can only be caught by measuring $T_1$ directly.

In [ ]:
reps = ba.coherence_repeats(T1T2_1850)
n = len(reps["Q2"]["T1"])
keep = np.array([i not in DIP_EXCLUDE["Q2"] for i in range(n)])
fig, axes = plt.subplots(3, 1, figsize=(6.4, 5.0), sharex=True, constrained_layout=True)
for ax, k, lab in zip(axes, ["T1", "T2echo", "T2*"], ["$T_1$ (µs)", "$T_2^{echo}$ (µs)", "$T_2^*$ (µs)"]):
    ax.axvspan(min(DIP_EXCLUDE["Q2"]) - 0.5, max(DIP_EXCLUDE["Q2"]) + 0.5, color=ORANGE, alpha=0.12, lw=0)
    for q, col, mk in [("Q1", BLUE, "o"), ("Q2", ORANGE, "s")]:
        ax.plot(np.arange(n), reps[q][k], mk + "-", color=col, ms=4, lw=0.8, label=q)
    v = reps["Q2"][k]
    ax.axhline(np.nanmean(v), color=REF, ls="--", lw=1)
    ax.axhline(np.nanmean(v[keep]), color=ORANGE, ls="--", lw=1)
    ax.set_ylabel(lab, fontsize=8); ax.set_ylim(0, None)
    ax.grid(axis="x", visible=False)
axes[0].text(np.mean(list(DIP_EXCLUDE["Q2"])), 3, "excluded (Q2 dip)", ha="center", color=INK2, fontsize=7.5)
axes[0].legend(loc="lower right", ncol=2, fontsize=7.5)
axes[-1].set_xlabel("repeat (measurement order, ≤ 30 s each), 18:50 run")
axes[-1].set_xticks(range(0, n, 2))
save(fig, "fig11_q2_dip_1850")
plt.show()

c_all = ba.coherence_times(T1T2_1850, drop_tls=False)
c_ex = T12["0924_2"]
for k in ["T1", "T2echo", "T2*"]:
    print(f"Q2 {k:7s} all repeats {c_all['Q2'][k][0]:5.1f}±{c_all['Q2'][k][1]:.1f}   dip excluded {c_ex['Q2'][k][0]:5.1f}±{c_ex['Q2'][k][1]:.1f} us")

## Summary

* **Estimator.** Average the CM-corrected linear-inversion $\rho$ over runs, apply one MLE, and
  bootstrap over runs for the error bar. Averaging per-run MLE fidelities is biased low by
  1–1.6 % at the shot counts used here (Figures 3–4).
* **0924_3.** Its pooled F (≈ 0.996) is above the $T_1/T_2$ limit because its CMs, measured right
  after a readout recalibration, over-correct: raw data identical to 0924_1, and corrected with
  0924_1's CM it gives ≈ 0.991 (section 6). Not a real improvement.
* **Best sets (0924_1, 0924_2).** Pooled $F$ = 0.993 ± 0.001 for both, reproducible across two
  sets 15 min apart without recalibration, and at the $T_1/T_2$ limit of the sequence. Gate-calibration errors each cost < 0.05 % (Figures 6–7) and the run-to-run scatter
  is shot noise (Figure 9).
* **What the corrected number means.** The notebook's CM is measured with the same prep pulses
  and pre-readout timing as the tomography, so it removes readout error *and* the decay during
  the tomography segment: the corrected fidelity is effectively the fidelity of the state the
  $\sqrt{i\mathrm{SWAP}}$ produced. A pure-readout correction gives the lower bound (Figure 7).
  A miscalibrated $X_\pi$ also leaks into this correction, which inflates the July values.
* **Limits.** The remaining ~0.7 % is $T_1/T_2^{\rm echo}$ decay during the 0.6 µs sequence, dominated by
  Q2. Improvements now need better Q2 coherence or shorter pulses, not more calibration.
* **Housekeeping.** Save the per-run CM with every set (done from 0924_1 on); it makes the pooled
  estimate exact without the CM recovery of section 6.